# 12 - Word Embeddings with TensorFlow/Keras

Goal: Learn word embeddings using TensorFlow and Keras

Run with: conda activate tfenv

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

I0000 00:00:1779038983.179852   15861 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779038983.663283   15861 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779038989.447163   15861 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version: 2.21.0


In [2]:
vocab = ["el", "la", "gato", "perro", "come", "duerme", "en", "casa", "corre", "nada"]
vocab_size = len(vocab)
embed_dim = 3

word_to_idx = {word: idx for idx, word in enumerate(vocab)}
print(f"Vocab size: {vocab_size}")
print(f"Words: {vocab}")
print(f"Word to index: {word_to_idx}")

Vocab size: 10
Words: ['el', 'la', 'gato', 'perro', 'come', 'duerme', 'en', 'casa', 'corre', 'nada']
Word to index: {'el': 0, 'la': 1, 'gato': 2, 'perro': 3, 'come': 4, 'duerme': 5, 'en': 6, 'casa': 7, 'corre': 8, 'nada': 9}


In [3]:
print("=== Method 1: tf.keras.layers.Embedding ===")
embedding_layer = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)

gato_idx = word_to_idx["gato"]
gato_embedding = embedding_layer(tf.constant(gato_idx))
print(f"Embedding for 'gato': {gato_embedding.numpy()}")

=== Method 1: tf.keras.layers.Embedding ===
Embedding for 'gato': [ 0.00100024 -0.01387192 -0.00747047]


E0000 00:00:1779038997.312540   15861 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1779038997.355336   15861 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [4]:
print("=== Method 2: Dense layer (manual) ===")
dense_layer = layers.Dense(embed_dim, use_bias=False)

print("\nBoth methods learn a weight matrix of shape:", embedding_layer.weights[0].shape)

=== Method 2: Dense layer (manual) ===

Both methods learn a weight matrix of shape: (10, 3)


In [5]:
import plotly.express as px
import pandas as pd

all_embeddings = embedding_layer(np.arange(vocab_size)).numpy()

df = pd.DataFrame(all_embeddings, columns=['x', 'y', 'z'])
df['word'] = vocab

fig = px.scatter_3d(df, x='x', y='y', z='z', text='word',
                    title='Word Embeddings (Random - TensorFlow)')
fig.update_traces(marker=dict(size=8), textposition='top center')
fig.show()

In [6]:
print("=== Training embeddings ===")

pairs = [
    (0, 2),
    (0, 3),
    (2, 3),
    (2, 8),
    (3, 8),
    (3, 9),
    (4, 5),
    (6, 7),
]

optimizer = keras.optimizers.Adam(learning_rate=0.1)
loss_fn = keras.losses.MeanSquaredError()

for epoch in range(500):
    total_loss = 0
    for word_idx, context_idx in pairs:
        word = tf.constant([word_idx])
        context = tf.constant([context_idx])

        with tf.GradientTape() as tape:
            word_emb = embedding_layer(word)
            context_emb = embedding_layer(context)
            loss = loss_fn(word_emb, context_emb * 0.5)

        gradients = tape.gradient(loss, embedding_layer.trainable_weights)
        optimizer.apply_gradients(zip(gradients, embedding_layer.trainable_weights))
        total_loss += loss

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss.numpy():.4f}")

=== Training embeddings ===
Epoch 0, Loss: 0.0676
Epoch 100, Loss: 0.0053
Epoch 200, Loss: 0.1480
Epoch 300, Loss: 0.0406
Epoch 400, Loss: 0.0043


In [7]:
trained_embeddings = embedding_layer(np.arange(vocab_size)).numpy()

df_trained = pd.DataFrame(trained_embeddings, columns=['x', 'y', 'z'])
df_trained['word'] = vocab

fig = px.scatter_3d(df_trained, x='x', y='y', z='z', text='word',
                    title='Word Embeddings (Trained - TensorFlow)')
fig.update_traces(marker=dict(size=8, color='red'), textposition='top center')
fig.show()

# Extra: Learn relationships from a real text

In [8]:
# Sample text in Spanish
text = """
El gato duerme en la casa. El perro corre en el jardin. 
La gata come pescado. El perro come carne.
El nino juega con la pelota. La nina lee el libro.
El pajaro vuela en el cielo. El pez nada en el agua.
La madre cocina la cena. El padre trabaja en la oficina.
El perro ladra. El gato maulla. El pajaro canta.
"""

# Tokenize and build vocabulary
words = text.lower().split()
words = [w.strip('.,') for w in words]
unique_words = list(set(words))
print(f"Text: {len(words)} words")
print(f"Unique words: {len(unique_words)}")
print(unique_words)

Text: 63 words
Unique words: 35
['jardin', 'gato', 'pajaro', 'agua', 'pescado', 'la', 'libro', 'canta', 'pelota', 'lee', 'corre', 'pez', 'padre', 'madre', 'nada', 'nino', 'nina', 'cocina', 'juega', 'duerme', 'trabaja', 'maulla', 'come', 'gata', 'con', 'carne', 'casa', 'el', 'vuela', 'perro', 'cielo', 'en', 'cena', 'oficina', 'ladra']


In [9]:
# Create word to index
text_vocab = {word: idx for idx, word in enumerate(unique_words)}
text_vocab_size = len(text_vocab)
text_embed_dim = 2

print(f"Vocabulary: {text_vocab}")

Vocabulary: {'jardin': 0, 'gato': 1, 'pajaro': 2, 'agua': 3, 'pescado': 4, 'la': 5, 'libro': 6, 'canta': 7, 'pelota': 8, 'lee': 9, 'corre': 10, 'pez': 11, 'padre': 12, 'madre': 13, 'nada': 14, 'nino': 15, 'nina': 16, 'cocina': 17, 'juega': 18, 'duerme': 19, 'trabaja': 20, 'maulla': 21, 'come': 22, 'gata': 23, 'con': 24, 'carne': 25, 'casa': 26, 'el': 27, 'vuela': 28, 'perro': 29, 'cielo': 30, 'en': 31, 'cena': 32, 'oficina': 33, 'ladra': 34}


In [10]:
# Create training pairs from context (window size = 1)
def create_pairs(words, window=1):
    pairs = []
    for i, word in enumerate(words):
        for j in range(max(0, i - window), min(len(words), i + window + 1)):
            if i != j:
                pairs.append((word, words[j]))
    return pairs

pairs = create_pairs(words)
print(f"Training pairs: {len(pairs)}")
print("Sample pairs:", pairs[:10])

Training pairs: 124
Sample pairs: [('el', 'gato'), ('gato', 'el'), ('gato', 'duerme'), ('duerme', 'gato'), ('duerme', 'en'), ('en', 'duerme'), ('en', 'la'), ('la', 'en'), ('la', 'casa'), ('casa', 'la')]


In [11]:
# Build embedding model with BATCH processing
text_embedding = layers.Embedding(input_dim=text_vocab_size, output_dim=text_embed_dim)
optimizer = keras.optimizers.Adam(learning_rate=0.05)
loss_fn = keras.losses.MeanSquaredError()

# Convert pairs to tensors and create batched dataset
word_indices = np.array([text_vocab[w] for w, c in pairs], dtype=np.int32)
context_indices = np.array([text_vocab[c] for w, c in pairs], dtype=np.int32)

# Create tf.data Dataset with batching
dataset = tf.data.Dataset.from_tensor_slices((word_indices, context_indices))
dataset = dataset.shuffle(buffer_size=len(pairs)).batch(32)

# Training with @tf.function for graph compilation (much faster)
@tf.function
def train_step(word_batch, context_batch):
    with tf.GradientTape() as tape:
        word_emb = text_embedding(word_batch)
        context_emb = text_embedding(context_batch)
        loss = loss_fn(word_emb, context_emb * 0.8)
    gradients = tape.gradient(loss, text_embedding.trainable_weights)
    optimizer.apply_gradients(zip(gradients, text_embedding.trainable_weights))
    return loss

print("Training with batch processing...")
import time

for epoch in range(500):
    start_time = time.time()
    total_loss = 0
    
    for word_batch, context_batch in dataset:
        loss = train_step(word_batch, context_batch)
        total_loss += loss
    
    epoch_time = time.time() - start_time
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss.numpy():.4f}, Time: {epoch_time:.3f}s")

Training with batch processing...
Epoch 0, Loss: 0.0196, Time: 0.811s
Epoch 100, Loss: 0.0002, Time: 0.015s
Epoch 200, Loss: 0.0016, Time: 0.016s
Epoch 300, Loss: 0.0100, Time: 0.015s
Epoch 400, Loss: 0.0000, Time: 0.015s


In [12]:
# Show learned embeddings
print("\n=== Learned Embeddings ===")
embeddings = text_embedding(np.arange(text_vocab_size)).numpy()

for word, idx in text_vocab.items():
    print(f"{word}: {embeddings[idx]}")


=== Learned Embeddings ===
jardin: [-0.00180241  0.00163588]
gato: [-0.00841334  0.00303621]
pajaro: [0.01217581 0.01079721]
agua: [-0.00601537 -0.0022084 ]
pescado: [0.00497549 0.00127683]
la: [-2.9605464e-05  1.2914285e-03]
libro: [-0.00829622 -0.00698051]
canta: [0.01362003 0.00348523]
pelota: [-0.03289802 -0.00087744]
lee: [ 0.00087167 -0.00310017]
corre: [-0.00957168 -0.0037573 ]
pez: [ 0.02108531 -0.007168  ]
padre: [ 0.00366248 -0.00349828]
madre: [-0.01054408 -0.00192277]
nada: [-0.0180337   0.00494161]
nino: [-0.0187849  -0.00648829]
nina: [-0.01342493  0.00013475]
cocina: [ 0.0060395  -0.00112221]
juega: [ 0.00086852 -0.00041   ]
duerme: [-0.00605777 -0.00227858]
trabaja: [0.0170515 0.0032896]
maulla: [-0.00980464 -0.00524886]
come: [-0.00708611  0.00144971]
gata: [-0.01031931 -0.00205654]
con: [ 0.00160663 -0.00117262]
carne: [-0.01119938 -0.00204118]
casa: [ 0.00954465 -0.00186441]
el: [0.011714   0.00549597]
vuela: [ 0.01251473 -0.00175968]
perro: [0.00827194 0.00108733]


In [13]:
# Visualize in 3D using Plotly
# Expand embeddings to 3D using a simple projection
embeddings_3d = np.column_stack([
    embeddings[:, 0],
    embeddings[:, 1],
    embeddings[:, 0] ** 2 + embeddings[:, 1] ** 2  # radial distance as 3rd dimension
])

df_3d = pd.DataFrame(embeddings_3d, columns=['x', 'y', 'z'])
df_3d['word'] = list(text_vocab.keys())

fig = px.scatter_3d(df_3d, x='x', y='y', z='z', text='word',
                    title='Word Embeddings from Text (3D)')
fig.update_traces(marker=dict(size=8), textposition='top center')
fig.show()

In [14]:
# Find similar words (closest embeddings)
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

def find_similar(word, top_n=3):
    word_idx = text_vocab[word]
    word_emb = embeddings[word_idx]
    
    similarities = []
    for w, idx in text_vocab.items():
        if w != word:
            sim = cosine_similarity(word_emb, embeddings[idx])
            similarities.append((w, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

print("=== Similar Words ===")
for word in ['gato', 'perro', 'come', 'casa']:
    if word in text_vocab:
        similar = find_similar(word)
        print(f"'{word}' is similar to: {similar}")

=== Similar Words ===
'gato' is similar to: [('nada', np.float32(0.99689066)), ('come', np.float32(0.98957306)), ('nina', np.float32(0.94398296))]
'perro' is similar to: [('trabaja', np.float32(0.9982077)), ('canta', np.float32(0.9928306)), ('pescado', np.float32(0.99274814))]
'come' is similar to: [('nada', np.float32(0.9978455)), ('gato', np.float32(0.98957306)), ('nina', np.float32(0.9816697))]
'casa' is similar to: [('cocina', np.float32(0.9999578)), ('cena', np.float32(0.99915904)), ('vuela', np.float32(0.9985846))]
